# MOVIE CREATOR — Kaggle All-in-One Studio (twin ng Colab notebook)

Pareho ito ng `colab_all_in_one_studio.ipynb`, pero para sa **Kaggle**:

| | |
| --- | --- |
| Accelerator | **GPU T4 x2** |
| Internet | **On** (Settings -> Internet -> On) |
| Base path | `/kaggle/working` |

**Honesty note:** walang ginagayang render dito. Kung bumagsak ang ComfyUI o kulang ang
weights, hihinto ang cell at ipapakita ang totoong error. Ang `SERVER_URL` ay ipi-print
lamang pagkatapos maabot ang `/android` nang **HTTP 200** sa public URL.

Bago magsimula: **Settings -> Accelerator = GPU T4 x2**, at **Internet = On**.

## 1. Suriin ang GPU

In [ ]:
import os, subprocess, sys

def sh(cmd, **kw):
    '''Run a shell command, echo it, and raise on failure.'''
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True, **kw)

BASE = "/kaggle/working"

smi = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
if smi.returncode != 0:
    sys.exit(
        "WALANG GPU.\n"
        "Settings -> Accelerator -> GPU T4 x2, tapos i-Run all muli."
    )
print(smi.stdout[:1200])

gpus = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True,
).stdout.strip().splitlines()
print("GPUs:", gpus)
if not any("T4" in g for g in gpus):
    print(f"PAALALA: inaasahan ang T4, nakuha: {gpus}. Pwede pa ring tumakbo kung sapat ang VRAM.")

# Kaggle has no internet unless the notebook setting is On.
net = subprocess.run("curl -s -o /dev/null -w '%{http_code}' --max-time 15 https://pypi.org",
                     shell=True, capture_output=True, text=True).stdout.strip()
if net != "200":
    sys.exit(
        "WALANG INTERNET.\n"
        "Settings -> Internet -> On, tapos i-Run all muli. Kailangan ito para sa "
        "Node, npm, ComfyUI at sa weights."
    )
print("Internet: ok")

## 2. Mga secret (opsyonal)

Kaggle: **Add-ons -> Secrets**, tapos i-attach sa notebook. Lahat opsyonal.

| Secret | Para saan |
| --- | --- |
| `GH_TOKEN` o `GITHUB_TOKEN` | Clone ng Private repo (kung Private na ito) |
| `GEMINI_API_KEY` | Gemini text/image path |
| `CF_TUNNEL_TOKEN` | Named Cloudflare tunnel |
| `CF_PUBLIC_URL` | Public hostname ng named tunnel, hal. `https://studio.example.com` |

Kapag wala sa Kaggle Secrets, hinahanap din sa environment variables. Wala sa mga
value ang ipi-print — presensya lang.

In [ ]:
import os

def get_secret(name):
    '''Kaggle Secrets first, then the process environment, else None.'''
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or None

GH_TOKEN = get_secret("GH_TOKEN") or get_secret("GITHUB_TOKEN")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
CF_TUNNEL_TOKEN = get_secret("CF_TUNNEL_TOKEN")
CF_PUBLIC_URL = get_secret("CF_PUBLIC_URL")

for name, value in [
    ("GH_TOKEN/GITHUB_TOKEN", GH_TOKEN),
    ("GEMINI_API_KEY", GEMINI_API_KEY),
    ("CF_TUNNEL_TOKEN", CF_TUNNEL_TOKEN),
    ("CF_PUBLIC_URL", CF_PUBLIC_URL),
]:
    print(f"{name:24s} {'meron' if value else 'wala'}")

if not GH_TOKEN:
    print("\nPaalala: kung Private na ang repo, hihinto ang clone sa susunod na cell.")

## 3. Node.js 22+, clone, at build

Ang token ay ipinapasa kay git gamit ang `http.extraheader` sa pamamagitan ng
**environment variable** (`GIT_AUTH_B64`). Dahil dito:

* wala ang token sa `ps` (hindi ito argument ng process), at
* wala rin ito sa naipi-print na command — `${GIT_AUTH_B64}` lang ang lumalabas.

Kaggle images ship an old Node, so 22 is installed from the official tarball into
`/kaggle/working/.tools` (no root needed, survives the session).

In [ ]:
import os, base64, subprocess, sys

TOOLS = f"{BASE}/.tools"
os.makedirs(TOOLS, exist_ok=True)

def node_major():
    try:
        out = subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip()
        return int(out.lstrip("v").split(".")[0]) if out else 0
    except Exception:
        return 0

if node_major() < 22:
    NODE_VERSION = "v22.20.0"
    tarball = f"node-{NODE_VERSION}-linux-x64.tar.xz"
    if not os.path.isdir(f"{TOOLS}/node-{NODE_VERSION}-linux-x64"):
        sh(f"curl -fsSL https://nodejs.org/dist/{NODE_VERSION}/{tarball} -o {TOOLS}/{tarball}")
        sh(f"tar -xJf {TOOLS}/{tarball} -C {TOOLS}")
    os.environ["PATH"] = f"{TOOLS}/node-{NODE_VERSION}-linux-x64/bin:" + os.environ["PATH"]
else:
    print("Node 22+ na ang naka-install")

print("node:", subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip())
print("npm :", subprocess.run("npm -v", shell=True, capture_output=True, text=True).stdout.strip())
if node_major() < 22:
    sys.exit("Bumagsak ang Node 22 install — titingnan ang PATH sa itaas.")

In [ ]:
import os, base64, subprocess, sys

REPO_DIR = f"{BASE}/MOVIES"
REPO_URL = "https://github.com/Rex217-0/MOVIES.git"

# The token goes to git through an env var + http.extraheader, never as an
# argument, so it cannot be read from `ps` and never lands in the echoed command.
clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
auth = ""
if GH_TOKEN:
    clone_env["GIT_AUTH_B64"] = base64.b64encode(
        f"x-access-token:{GH_TOKEN}".encode()
    ).decode()
    auth = '-c http.extraheader="AUTHORIZATION: basic ${GIT_AUTH_B64}" '
    print("Token: meron — makakapag-clone kahit Private ang repo")
else:
    print("Token: wala — public clone lang")

def run(cmd, **kw):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True, **kw)

if not os.path.isdir(REPO_DIR):
    run(f"git {auth}clone --branch main --depth 1 {REPO_URL} {REPO_DIR}", env=clone_env)
else:
    run(f'git -C {REPO_DIR} {auth}fetch --depth 1 origin main', env=clone_env)
    run(f"git -C {REPO_DIR} reset --hard origin/main")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print(subprocess.run("git log -1 --oneline", shell=True, capture_output=True, text=True).stdout.strip())

# npm ci is reproducible and matches package-lock.json exactly.
run("npm ci --no-audit --no-fund", env=clone_env)
run("npm run build", env=clone_env)
if not os.path.isdir("dist"):
    sys.exit("Walang dist/ pagkatapos ng build — titingnan ang output sa itaas.")
print("dist/ ok")

## 4. ComfyUI, VideoHelperSuite, at cloudflared

In [ ]:
import os, subprocess, sys

COMFY = f"{BASE}/ComfyUI"

def run(cmd, cwd=None, **kw):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True, cwd=cwd, **kw)

if not os.path.isdir(COMFY):
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}")
else:
    print("ComfyUI already cloned")

run("pip install -q -r requirements.txt", cwd=COMFY)

# VHS_VideoCombine (mp4 output) comes from this custom node pack.
VHS = f"{COMFY}/custom_nodes/ComfyUI-VideoHelperSuite"
if not os.path.isdir(VHS):
    run(f"git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git {VHS}")
    run("pip install -q -r requirements.txt", cwd=VHS)
else:
    print("VideoHelperSuite already present")

CLOUDFLARED = f"{BASE}/cloudflared"
if not os.path.isfile(CLOUDFLARED):
    run(f"curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -o {CLOUDFLARED}")
    os.chmod(CLOUDFLARED, 0o755)
print(subprocess.run(f"{CLOUDFLARED} --version", shell=True, capture_output=True, text=True).stdout.strip())

## 5. Wan 2.1 1.3B weights — direkta sa tamang folder

Ang `hf_hub_download(local_dir=...)` ay **giniit ang remote subpath**. Dahil ang
files sa `Comfy-Org/Wan-2.1` ay nasa `split_files/...`, ang dating code ay
naglalagay ng:

```
models/vae/split_files/vae/wan_2.1_vae.safetensors   <- mali, hindi ito makikita
```

Kaya binabawi ng cell na ito ang file pataas sa `models/vae/` at inaalis ang
`split_files/` folder.

In [ ]:
import os, shutil, sys
from huggingface_hub import hf_hub_download

COMFY = f"{BASE}/ComfyUI"
REPO = "Comfy-Org/Wan-2.1"

# remote path -> (local folder, expected minimum bytes)
FILES = {
    "split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors": ("diffusion_models", 2_800_000_000),
    "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors": ("text_encoders", 6_700_000_000),
    "split_files/vae/wan_2.1_vae.safetensors": ("vae", 250_000_000),
}

for remote, (subdir, min_bytes) in FILES.items():
    dest_dir = f"{COMFY}/models/{subdir}"
    os.makedirs(dest_dir, exist_ok=True)
    final = os.path.join(dest_dir, os.path.basename(remote))

    if os.path.isfile(final) and os.path.getsize(final) >= min_bytes:
        print(f"{final}  {os.path.getsize(final)/1e9:.2f} GB  (meron na)")
        continue

    path = hf_hub_download(repo_id=REPO, filename=remote, local_dir=dest_dir)

    # hf_hub_download keeps split_files/... under dest_dir; flatten it.
    if os.path.abspath(path) != os.path.abspath(final):
        os.replace(path, final)
        parent = os.path.dirname(path)
        while os.path.abspath(parent) != os.path.abspath(dest_dir):
            try:
                os.rmdir(parent)
            except OSError:
                break
            parent = os.path.dirname(parent)

    size = os.path.getsize(final)
    if size < min_bytes:
        sys.exit(f"{final} ay {size} bytes lang — hindi kumpleto ang download.")
    print(f"{final}  {size/1e9:.2f} GB")

# Nothing should be left behind under split_files/.
stray = []
for subdir, _ in FILES.values():
    nested = f"{COMFY}/models/{subdir}/split_files"
    if os.path.isdir(nested):
        stray.append(nested)
if stray:
    sys.exit(f"May natira pang split_files/: {stray}")
print("Walang split_files/ — tama ang layout")

## 6. Patakbuhin ang ComfyUI sa `127.0.0.1:8188`

In [ ]:
import os, subprocess, sys, time, urllib.request

COMFY = f"{BASE}/ComfyUI"
COMFY_LOG = f"{BASE}/comfyui.log"

comfy_proc = subprocess.Popen(
    "python main.py --listen 127.0.0.1 --port 8188 > " + COMFY_LOG + " 2>&1",
    shell=True, cwd=COMFY,
)

def wait_http(url, timeout, label):
    deadline = time.time() + timeout
    last = ""
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=10) as r:
                if r.status == 200:
                    print(f"{label} -> 200")
                    return True
                last = f"HTTP {r.status}"
        except Exception as e:
            last = str(e)
        time.sleep(5)
    print(f"{label} FAILED: {last}")
    return False

if not wait_http("http://127.0.0.1:8188/system_stats", 420, "ComfyUI"):
    print(open(COMFY_LOG).read()[-4000:])
    sys.exit("Bumagsak ang ComfyUI — tingnan ang log sa itaas.")

In [ ]:
import json, urllib.request

with urllib.request.urlopen("http://127.0.0.1:8188/object_info") as r:
    info = json.load(r)

for node in ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode",
             "EmptyHunyuanLatentVideo", "KSampler", "VAEDecode", "VHS_VideoCombine"]:
    assert node in info, f"missing node {node} (i-restart ang ComfyUI pagkatapos mag-install ng custom nodes)"
print("all workflow nodes present")

unets = info["UNETLoader"]["input"]["required"]["unet_name"][0]
clips = info["CLIPLoader"]["input"]["required"]["clip_name"][0]
vaes = info["VAELoader"]["input"]["required"]["vae_name"][0]
print("unet :", unets)
print("clip :", clips)
print("vae  :", vaes)

# The loaders list what they can actually see on disk. If the weights were
# downloaded into the wrong folder, they are missing here.
for got, want in [(unets, "wan2.1_t2v_1.3B_fp16.safetensors"),
                  (clips, "umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
                  (vaes, "wan_2.1_vae.safetensors")]:
    assert want in got, f"{want} hindi makita ng ComfyUI — mali ang folder"
print("Nakikita ng ComfyUI ang lahat ng weights")

## 7. Patakbuhin ang app sa `127.0.0.1:3000`

`NODE_ENV=production` ay nasa **command**, hindi sa `.env`, kaya laging tama ang mode
kahit may makaligtaang `.env`. `ALLOW_PAID_CLOUD=0` para hindi mapunta sa bayad na
provider. `APP_AUTH_TOKEN` ay kailangan sa production — hihinto ang server kung wala.

In [ ]:
import os, secrets, subprocess, sys, time, urllib.request

APP_AUTH_TOKEN = secrets.token_urlsafe(32)
PORT = 3000

# dotenv does not override variables that already exist in the environment, so
# these shell-level values win over anything stale in a checked-in .env.
app_env = os.environ.copy()
app_env.update({
    "NODE_ENV": "production",
    "PORT": str(PORT),
    "APP_AUTH_TOKEN": APP_AUTH_TOKEN,
    "ALLOW_PAID_CLOUD": "0",
    "COMFYUI_URL": "http://127.0.0.1:8188",
    "PREFERRED_RENDER_ENGINE": "comfyui",
    "COMFYUI_CHECKPOINT": "wan2.1_t2v_1.3B_fp16.safetensors",
})
if GEMINI_API_KEY:
    app_env["GEMINI_API_KEY"] = GEMINI_API_KEY

APP_LOG = f"{BASE}/app.log"
app_proc = subprocess.Popen(
    f"cd {REPO_DIR} && npm start > {APP_LOG} 2>&1",
    shell=True, env=app_env,
)

def wait_http(url, timeout, label):
    deadline = time.time() + timeout
    last = ""
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=10) as r:
                if r.status == 200:
                    print(f"{label} -> 200")
                    return True
                last = f"HTTP {r.status}"
        except Exception as e:
            last = str(e)
        time.sleep(4)
    print(f"{label} FAILED: {last}")
    return False

if not wait_http(f"http://127.0.0.1:{PORT}/", 300, "App"):
    print(open(APP_LOG).read()[-4000:])
    sys.exit("Bumagsak ang app — tingnan ang log sa itaas.")
print("App tumatakbo sa production mode")

## 8. Cloudflare tunnel + DNS retry

Bagong `trycloudflare.com` hostname ay hindi agad nire-resolve ng DNS. Kung
susubukan mong i-fetch agad, `gaierror: [Errno -2] Name or service not known` ang
lalabas — hindi ito sira ng tunnel. Kaya may **retry loop** na naghihintay hanggang
ma-resolve ang DNS bago mag-verify.

Kapag may `CF_TUNNEL_TOKEN` **at** `CF_PUBLIC_URL`, named tunnel ang gagamitin.
Kung wala, quick tunnel (bagong URL sa bawat restart).

In [ ]:
import os, re, socket, subprocess, sys, time, urllib.error, urllib.request

TUNNEL_LOG = f"{BASE}/cloudflared.log"
SERVER_URL = None

def resolve_ok(host, timeout=8):
    '''True once DNS actually resolves the host. This is the gaierror guard.'''
    try:
        socket.getaddrinfo(host, 443)
        return True
    except socket.gaierror:
        return False

if CF_TUNNEL_TOKEN and CF_PUBLIC_URL:
    # TUNNEL_TOKEN is passed through the environment, not argv, so the token
    # never shows up in `ps`.
    print("Named tunnel (CF_TUNNEL_TOKEN + CF_PUBLIC_URL)")
    tun_env = os.environ.copy()
    tun_env["TUNNEL_TOKEN"] = CF_TUNNEL_TOKEN
    tunnel_proc = subprocess.Popen(
        # --no-autoupdate belongs to the `tunnel` command, before `run`.
        f"{CLOUDFLARED} tunnel --no-autoupdate run --url http://127.0.0.1:{PORT} "
        f"> {TUNNEL_LOG} 2>&1",
        shell=True, env=tun_env,
    )
    SERVER_URL = CF_PUBLIC_URL.strip().rstrip("/")
    if not SERVER_URL.startswith("http"):
        SERVER_URL = "https://" + SERVER_URL
else:
    print("Quick tunnel (walang CF_TUNNEL_TOKEN + CF_PUBLIC_URL)")
    tunnel_proc = subprocess.Popen(
        f"{CLOUDFLARED} tunnel --url http://127.0.0.1:{PORT} --no-autoupdate "
        f"> {TUNNEL_LOG} 2>&1",
        shell=True,
    )
    deadline = time.time() + 180
    while time.time() < deadline and SERVER_URL is None:
        if os.path.exists(TUNNEL_LOG):
            found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
            if found:
                SERVER_URL = found[-1]
                break
        time.sleep(3)
    if not SERVER_URL:
        print(open(TUNNEL_LOG).read()[-2000:])
        sys.exit("Walang trycloudflare.com URL — tingnan ang log sa itaas.")

host = SERVER_URL.split("://", 1)[-1].split("/")[0]
print("Host:", host)

# DNS first: a fresh hostname often fails to resolve for the first few seconds.
deadline = time.time() + 300
while time.time() < deadline:
    if resolve_ok(host):
        print("DNS ok")
        break
    time.sleep(5)
else:
    print(open(TUNNEL_LOG).read()[-1500:])
    sys.exit(f"Hindi nag-resolve ang {host} sa loob ng 300s.")

# Then the real check: /android must answer 200 through the public edge.
deadline = time.time() + 300
last = ""
while time.time() < deadline:
    try:
        with urllib.request.urlopen(f"{SERVER_URL}/android", timeout=20) as r:
            if r.status == 200:
                print(f"Public {SERVER_URL}/android -> 200")
                break
            last = f"HTTP {r.status}"
    except urllib.error.HTTPError as e:
        last = f"HTTP {e.code}"
    except Exception as e:
        last = f"{type(e).__name__}: {e}"
    time.sleep(5)
else:
    print(open(TUNNEL_LOG).read()[-1500:])
    sys.exit(f"Bukas ang tunnel pero hindi maabot ang /android: {last}")

## 9. Ito ang kailangan mo

In [ ]:
print("=" * 64)
print("SERVER_URL    :", SERVER_URL)
print("APP_AUTH_TOKEN:", APP_AUTH_TOKEN)
print("=" * 64)
print()
print("Ilagay ito sa Server Setup ng APK:")
print(f"  {SERVER_URL}")
print()
print("Ilagay ito sa .env ng Movie Creator:")
print(f"  APP_AUTH_TOKEN={APP_AUTH_TOKEN}")
print(f"  COMFYUI_URL=http://127.0.0.1:8188")
print(f"  COMFYUI_VIDEO_WORKFLOW=./workflows/comfyui_video_api.json")
print(f"  COMFYUI_CHECKPOINT=wan2.1_t2v_1.3B_fp16.safetensors")
print(f"  PREFERRED_RENDER_ENGINE=comfyui")
print(f"  ALLOW_PAID_CLOUD=0")
print()
print("Paalala: habang tumatakbo ang session, bukas ang URL. Kapag nag-stop ang")
print("Kaggle session, patay din ang URL — bagong URL sa susunod na run.")

## Troubleshooting

| Sintomas | Dahilan at ayos |
| --- | --- |
| `gaierror` / `Name or service not known` | Normal sa bagong hostname. Ang retry loop ang humahawak; kung lumampas sa 300s, tingnan ang `cloudflared.log`. |
| `Walang GPU` | Settings -> Accelerator -> GPU T4 x2. |
| `WALANG INTERNET` | Settings -> Internet -> On. |
| `APP_AUTH_TOKEN must be set` | Hindi umabot ang env sa `npm start`. Tingnan ang `/kaggle/working/app.log`. |
| `missing node VHS_VideoCombine` | Hindi na-install ang VideoHelperSuite. I-rerun ang Section 4, tapos i-restart ang ComfyUI. |
| `hindi makita ng ComfyUI` ang weights | May `split_files/` pa. I-rerun ang Section 5 — inaalis na nito ang nested folder. |
| Clone failed sa Private repo | Kulang ang `GH_TOKEN`/`GITHUB_TOKEN` sa Kaggle Secrets. |